# Robust NLP Framework for Automated Financial Transaction Extraction and Expense Categorization

**School of Computer Science and Engineering — RV University**  
**Team ID: 10**  
- Preksha P V (`1RUA24CSE0331`)
- Reshma K P (`1RUA24CSE0363`)
- Prathik K B (`1RUA24CSE0327`)
- Rushil Ravi Reddy (`1RUA24CSE0380`)

**Guides:** Prof. Khushboo Pandey & Prof. Priyanka S Shivaramaiah  
**Dataset:** [`Ranjit0034/finee-dataset`](https://huggingface.co/datasets/Ranjit0034/finee-dataset) (152,000+ Indian Banking & UPI messages across 6 Indian languages + Code-Mixed Augmentation)

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_or_build_dataset
from src.preprocessor import preprocess_record
from src.extractor import FinancialEntityExtractor
from src.categorizer import ClassicalMLCategorizer, HybridTransactionPipeline
from src.evaluator import evaluate_entity_extraction, evaluate_categorization_models

sns.set_theme(style="whitegrid")

## 1. Load & Inspect FinEE Dataset + Multilingual / Code-Mixed Augmentation

In [ ]:
df_train, df_valid, df_test = load_or_build_dataset()
print(f"Train: {len(df_train):,} | Valid: {len(df_valid):,} | Test: {len(df_test):,}")
df_train[["raw_sms", "amount", "type", "bank", "merchant", "category", "synopsis_category", "language", "is_code_mixed"]].head(10)

## 2. Exploratory Data Analysis (Languages & Expense Categories)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_train["category"].value_counts().plot(kind="barh", ax=axes[0], color="#2563eb", title="Distribution of Expense Categories")
df_train["language"].value_counts().plot(kind="bar", ax=axes[1], color="#10b981", title="Distribution of Languages & Code-Mixed SMS")
plt.tight_layout()
plt.show()

## 3. Multilingual & Code-Mixed Financial Entity Extraction Demo

In [ ]:
pipeline = HybridTransactionPipeline()

sample_messages = [
    "HDFC Bank: Rs.7,361.93 debited from A/c XX1489 on 03-11-25. VPA netmeds@icici. UPI Ref: 549264265572.",
    "HDFC Bank Alert: Aapke A/c XX4248 se Rs.485.50 kat gaya for Swiggy payment on 28-04-2025 to swiggy@ybl. UPI Ref: 430298465323.",
    "SBI: ನಿಮ್ಮ ಖಾತೆ XX5059 ನಿಂದ Rs.530.87 ಡೆಬಿಟ್ ಆಗಿದೆ. ದಿನಾಂಕ 02-02-2025. zepto@ybl. Ref: 000359529957",
    "SBI: INR 65,000.00 paid from A/c XX4821 on 14-08-2025 towards RV University Tuition Fee. VPA rvuniversity@sbi. Ref: 839201746512."
]

for msg in sample_messages:
    res = pipeline.process_message(msg)
    print("-" * 90)
    print("SMS             :", res["raw_sms"])
    print(f"Extracted       : Amount=Rs.{res['amount']} | Type={res['type']} | Bank={res['bank']} | Acct={res['account']} | Date={res['date']}")
    print(f"Party & Mode    : Merchant={res['merchant']} | VPA={res['vpa']} | Mode={res['payment_mode']}")
    print(f"Categorization  : {res['synopsis_category']} ({res['category']}) [Conf: {res['confidence']*100:.1f}%] | Lang={res['language']} (Code-Mixed={res['is_code_mixed']})")
    print(f"XAI Rationales  : {res['explanations']}")

## 4. Model Evaluation Summary (Accuracy, Precision, Recall, F1-Score)

In [ ]:
import json
with open("../outputs/evaluation_report.json") as f:
    report = json.load(f)

print("=== EXPENSE CATEGORIZATION MODEL BENCHMARK ===")
display(pd.DataFrame(report["expense_categorization"]["overall_models"]).T)

print("\n=== FINANCIAL ENTITY EXTRACTION (NER + REGEX) BENCHMARK ===")
display(pd.DataFrame(report["entity_extraction"]).T)

print("\n=== MULTILINGUAL & CODE-MIXED PERFORMANCE BREAKDOWN ===")
display(pd.DataFrame(report["expense_categorization"]["language_breakdown"]).T)